In [0]:
import numpy as np
import pandas as pd
import json as _json

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from xgboost import XGBRanker, XGBClassifier, XGBRegressor

In [0]:
MASTER_PATH = "workspace.default.f1_ml_lap_dataset"

df = spark.table(MASTER_PATH)

In [0]:
# Podium/win ranking models from phase_3_ranking.ipynb. These are XGBRanker, not
# XGBClassifier -- phase_3_ranking.ipynb reframed podium/win prediction as
# learning-to-rank, grouped per race/lap-snapshot (see CLAUDE.md). Loading
# them into an XGBClassifier instance, as this stub originally did, loads
# the saved booster into a wrapper with the wrong objective/interface
# (predict_proba doesn't exist on the trained booster's actual task type).
pre_model = XGBRanker()
pre_model.load_model("../models/pre_model.json")

live_model = XGBRanker()
live_model.load_model("../models/live_model.json")

### Strategy simulator (Plan B)

Artifacts trained in `phase_3_strategy.ipynb` — a stint-level compound classifier (`m1`), next-pit-lap regressor (`m2`), and win-probability classifier (`m3`), plus a preprocessing transform (`build_feature_vector`, pure NumPy — not a `pyspark.ml.Pipeline`; see below) that feeds them. Persisted to a Unity Catalog Volume rather than a workspace-relative path so both this notebook and the Databricks App (`app/`) can load the exact same files — see that notebook's model-saving cell for why `../models/` alone isn't reachable from either.

`pre_model`/`live_model` above are not combined with the strategy simulator here — that fusion is still open (see CLAUDE.md).

In [0]:
STRATEGY_VOLUME_DIR = "/Volumes/workspace/default/f1_data/strategy_models"

# NOT loaded via pyspark.ml.PipelineModel.load(): phase_3_strategy.ipynb no
# longer fits a pyspark.ml.Pipeline at all (it kept hitting
# [CONNECT_ML.MODEL_SIZE_OVERFLOW_EXCEPTION] under Spark Connect/serverless
# compute even after severing upstream join lineage, so preprocessing is
# computed natively via plain Spark aggregations instead -- see that
# notebook's preprocessing cells). preproc_params.json is what's persisted
# now; build_feature_vector() below reimplements the same transform in
# pure NumPy, mirroring app.py's copy exactly (keep both in sync).
with open(f"{STRATEGY_VOLUME_DIR}/preproc_params.json") as f:
    preproc_params = _json.load(f)

m1 = XGBClassifier()
m1.load_model(f"{STRATEGY_VOLUME_DIR}/m1_compound_classifier.json")

m2 = XGBRegressor()
m2.load_model(f"{STRATEGY_VOLUME_DIR}/m2_pitlap_regressor.json")

m3 = XGBClassifier()
m3.load_model(f"{STRATEGY_VOLUME_DIR}/m3_win_classifier.json")

with open(f"{STRATEGY_VOLUME_DIR}/compound_label_map.json") as f:
    COMPOUND_IDX_TO_NAME = {int(k): v for k, v in _json.load(f).items()}

driver_profile = spark.table("workspace.default.f1_strategy_driver_profile")
constructor_profile = spark.table("workspace.default.f1_strategy_constructor_profile")
circuit_profile = spark.table("workspace.default.f1_strategy_circuit_profile")

CAT_FEATURES = ["Circuit", "TeamName", "Driver"]
TYRE_LIFE_LIMITS = {"SOFT": 25, "MEDIUM": 40, "HARD": 55, "INTERMEDIATE": 30, "WET": 40}


def build_feature_vector(row_dict, driver, team, circuit, preproc_params):
    """Pure-NumPy reimplementation of the StringIndexer -> OneHotEncoder ->
    VectorAssembler -> StandardScaler transform phase_3_strategy.ipynb now
    computes natively. Mirrors app.py's _build_feature_vector() exactly --
    keep both in sync if this changes.
    """
    cat_values = {"Circuit": circuit, "TeamName": team, "Driver": driver}
    vec = [float(row_dict[nf]) for nf in preproc_params["numeric_features"]]
    for c in preproc_params["cat_features"]:
        labels = preproc_params["cat_labels"][c]
        n = len(labels)
        oh = [0.0] * n
        val = cat_values[c]
        if val in labels:
            oh[labels.index(val)] = 1.0
        vec.extend(oh)
    x = np.array(vec, dtype=float)
    mean = np.array(preproc_params["scaler_mean"])
    std = np.array(preproc_params["scaler_std"])
    out = np.where(std != 0, (x - mean) / np.where(std == 0, 1.0, std), 0.0)
    return out.reshape(1, -1)

In [0]:
def predict_strategy(
    driver: str,
    team: str,
    year: int,
    circuit: str,
    quali_position: int,
    total_laps: int = 57,
    num_pit_stops: int = 1,
    verbose: bool = True,
) -> dict:
    """
    Predict a pit stop strategy for one driver in one hypothetical race,
    enforcing tyre-life physics constraints and a user-chosen stop count.

    Adapted from phase_3_strategy.ipynb's predict_strategy —
    same modeling logic, sourced from the Volume/UC-table artifacts above
    instead of that notebook's own in-session Spark state. The
    win-probability calibration blend was rebalanced in both places to fix
    a saturation bug (see that notebook's win-probability cell for detail).
    Preprocessing uses build_feature_vector() (pure NumPy) rather than a
    Spark ML Pipeline, matching phase_3_strategy.ipynb and app.py — see
    this notebook's model-loading cell for why.

    Returns {"strategy": [...], "win_probability": float}.
    """

    def get_profile(sdf_local, col, val, year_local):
        row = sdf_local.filter((F.col(col) == val) & (F.col("Year") == year_local)).first()
        if row is None:
            row = sdf_local.filter(F.col(col) == val).orderBy(F.desc("Year")).first()
        return row

    def safe(row, key, default=10.0):
        try:
            return float(row[key]) if row and row[key] is not None else default
        except Exception:
            return default

    drv_row = get_profile(driver_profile, "Driver", driver, year)
    car_row = get_profile(constructor_profile, "TeamName", team, year)
    cir_row = circuit_profile.filter(F.col("Circuit") == circuit).first()

    drv_avg_pos   = safe(drv_row, "DriverAvgPos", 10.0)
    drv_avg_quali = safe(drv_row, "DriverAvgQuali", quali_position)
    drv_win_rate  = safe(drv_row, "DriverWinRate", 0.05)
    drv_pts_rate  = safe(drv_row, "DriverPointsRate", 0.5)
    drv_pod_rate  = safe(drv_row, "DriverPodiumRate", 0.15)
    drv_consist   = safe(drv_row, "DriverConsistency", 4.0)
    car_avg_pos   = safe(car_row, "CarAvgPos", 10.0)
    car_avg_quali = safe(car_row, "CarAvgQuali", quali_position)
    car_win_rate  = safe(car_row, "CarWinRate", 0.05)
    car_pod_rate  = safe(car_row, "CarPodiumRate", 0.15)
    car_pts_rate  = safe(car_row, "CarPointsRate", 0.5)
    cir_avg_comp  = safe(cir_row, "CircuitAvgCompoundRank", 3.0)
    cir_avg_temp  = safe(cir_row, "CircuitAvgTrackTemp", 30.0)

    # Threat score: sum of win rates for every driver starting ahead on the
    # grid. Sourced from the base cleaned lap dataset rather than the
    # experimental notebook's own in-session `sdf` — QualiPosition is
    # race-constant per driver, so the values are identical either way.
    year_drivers = (
        spark.table("workspace.default.f1_cleaned_lap_dataset")
        .filter(F.col("Year") == year)
        .select("Driver", "QualiPosition")
        .dropDuplicates()
    )
    threat_df = year_drivers.filter(F.col("QualiPosition") < quali_position)

    threat_score = 0.0
    max_threat = 0.0
    for t_row in threat_df.collect():
        t_drv = get_profile(driver_profile, "Driver", t_row["Driver"], year)
        t_wr = safe(t_drv, "DriverWinRate", 0.0)
        threat_score += t_wr
        max_threat = max(max_threat, t_wr)

    strategy = []
    current_lap = 1
    current_pos = quali_position
    total_stints = num_pit_stops + 1
    prev_compound = None

    for stint_num in range(1, total_stints + 1):
        lap_frac = current_lap / total_laps

        row_dict = {
            "Stint": stint_num,
            "StintStartLap": current_lap,
            "StintLength": max(1, total_laps - current_lap),
            "StintStartFrac": lap_frac,
            "StintAvgLapTime": 95.0,
            "StintBestLap": 92.0,
            "QualiPosition": quali_position,
            "TotalLaps": total_laps,
            "NumStints": total_stints,
            "AvgGapToAhead": 1.5,
            "AvgDeltaToLeader": max(0, (current_pos - 1) * 1.2),
            "PositionAtStintEnd": current_pos,
            "AirTemp_C": 22.0,
            "TrackTemp_C": cir_avg_temp,
            "Humidity_pct": 40.0,
            "WindSpeed_kmh": 2.0,
            "DriverAvgPos": drv_avg_pos,
            "DriverAvgQuali": drv_avg_quali,
            "DriverWinRate": drv_win_rate,
            "DriverPointsRate": drv_pts_rate,
            "DriverPodiumRate": drv_pod_rate,
            "DriverConsistency": drv_consist,
            "CarAvgPos": car_avg_pos,
            "CarAvgQuali": car_avg_quali,
            "CarWinRate": car_win_rate,
            "CarPodiumRate": car_pod_rate,
            "CarPointsRate": car_pts_rate,
            "CircuitAvgCompoundRank": cir_avg_comp,
            "CircuitAvgStints": total_stints,
            "CircuitAvgTrackTemp": cir_avg_temp,
            "ThreatScoreAhead": threat_score,
            "MaxThreatAhead": max_threat,
            "FreshTyre": 1,
            "FinalPosition": current_pos,
            "IsWin": 0,
            "StintCompound": "SOFT",
            "NextPitLap": float(total_laps),
        }

        feat_arr = build_feature_vector(row_dict, driver, team, circuit, preproc_params)

        prob_vec = m1.predict_proba(feat_arr)[0]
        comp_prefs = [(COMPOUND_IDX_TO_NAME.get(i, "SOFT"), p) for i, p in enumerate(prob_vec)]
        comp_prefs.sort(key=lambda x: x[1], reverse=True)

        valid_compounds = [c[0] for c in comp_prefs if c[0] in ["SOFT", "MEDIUM", "HARD"]]

        # Pirelli rule: must use 2 different compounds across the race.
        if stint_num == total_stints and total_stints > 1:
            used_compounds = set(s["compound"] for s in strategy)
            if len(used_compounds) == 1:
                used_c = list(used_compounds)[0]
                valid_compounds = [c for c in valid_compounds if c != used_c]

        # Avoid repeating the same compound back-to-back.
        if len(valid_compounds) > 1 and prev_compound in valid_compounds:
            valid_compounds.remove(prev_compound)

        # 1-stop durability: SOFT is too risky for a 2-stint race, and one
        # stint must be HARD to survive the distance.
        if total_stints <= 2:
            if valid_compounds and valid_compounds[0] == "SOFT":
                hard_med = [c for c in valid_compounds if c in ["HARD", "MEDIUM"]]
                if hard_med:
                    valid_compounds = hard_med + [c for c in valid_compounds if c not in hard_med]
            if stint_num == 2 and "HARD" not in [s["compound"] for s in strategy]:
                if "HARD" in valid_compounds:
                    valid_compounds.remove("HARD")
                    valid_compounds.insert(0, "HARD")

        compound = valid_compounds[0] if valid_compounds else "SOFT"

        if stint_num == total_stints:
            next_pit = total_laps
        else:
            p_lap = int(round(float(m2.predict(feat_arr)[0])))
            max_laps_for_compound = TYRE_LIFE_LIMITS.get(compound, 30)
            p_lap = min(p_lap, current_lap + max_laps_for_compound)
            p_lap = max(p_lap, current_lap + 5)
            next_pit = p_lap

            remaining_laps = total_laps - next_pit
            remaining_stints = total_stints - stint_num
            if remaining_laps > remaining_stints * max(TYRE_LIFE_LIMITS.values()):
                next_pit = min(current_lap + max_laps_for_compound, total_laps - remaining_stints * 5)

        strategy.append({
            "stint": stint_num,
            "lap_start": current_lap,
            "lap_end": next_pit - 1 if next_pit < total_laps else total_laps,
            "compound": compound,
            "laps_on_tyre": (next_pit - 1 if next_pit < total_laps else total_laps) - current_lap + 1,
        })

        prev_compound = compound
        current_lap = next_pit

    # Win probability, calibrated against grid-position baseline + driver
    # win rate + threat from cars ahead (same inputs as the source notebook).
    wp_dict = dict(row_dict)
    wp_dict["StintCompound"] = strategy[0]["compound"]
    wp_arr = build_feature_vector(wp_dict, driver, team, circuit, preproc_params)
    raw_win_prob = float(m3.predict_proba(wp_arr)[0, 1])

    grid_baseline = (
        0.40 if quali_position == 1 else
        0.25 if quali_position == 2 else
        0.15 if quali_position == 3 else
        max(0.01, 0.10 - quali_position * 0.01)
    )
    driver_factor = min(1.0, drv_win_rate * 2.0)
    threat_penalty = max(0, threat_score - drv_win_rate) * 0.5
    # Weighted blend (weights sum to 1.0) so no single term can force
    # saturation on its own -- the old (raw*3.0)+grid+driver stacking could
    # exceed 1.0 before the 0.999 cap for almost any front-running or
    # decent-win-rate driver, making every prediction read ~99.9%.
    calibrated_prob = (0.55 * raw_win_prob) + (0.30 * grid_baseline) + (0.15 * driver_factor) - threat_penalty
    win_prob = max(0.001, min(0.999, calibrated_prob))

    if verbose:
        print(f"\n{'='*65}")
        print(f"  STRATEGY PREDICTION ({num_pit_stops}-Stop)")
        print(f"  Driver: {driver}  |  Team: {team}")
        print(f"  Circuit: {circuit}  |  Year: {year}  |  Grid: P{quali_position}")
        print(f"  Threat Ahead: {threat_score:.2f} (Max: {max_threat:.2f})")
        print(f"{'='*65}")
        print(f"  {'Stint':<8} {'Laps':<18} {'Compound':<14} {'Tyre Life'}")
        print(f"  {'-'*55}")
        for s in strategy:
            lap_range = f"Lap {s['lap_start']} - {s['lap_end']}"
            print(f"  {s['stint']:<8} {lap_range:<18} {s['compound']:<14} {s['laps_on_tyre']} laps")
        print(f"{'='*65}")
        print(f"  Win Probability  : {win_prob*100:.1f}%")
        print(f"{'='*65}")

    return {"strategy": strategy, "win_probability": win_prob}

In [0]:
# Demo — same example as phase_3_strategy.ipynb's own
# example cells, now running against the persisted Volume/UC-table
# artifacts instead of that notebook's in-session state.
result = predict_strategy(
    driver="HAM",
    team="Mercedes",
    year=2023,
    circuit="Monza",
    quali_position=1,
    total_laps=53,
    num_pit_stops=1,
)

### Battle prediction (Plan G)

`battle_model` trained in `phase_3_battles.ipynb` — binary overtake-probability classifier over pairs of drivers running within a gap threshold, plus `f1_circuit_overtake_prior` for the circuit-level feature.

In [0]:
BATTLE_VOLUME_DIR = "/Volumes/workspace/default/f1_data/battle_models"

battle_model = XGBClassifier()
battle_model.load_model(f"{BATTLE_VOLUME_DIR}/m_overtake_classifier.json")

circuit_overtake_prior = spark.table("workspace.default.f1_circuit_overtake_prior")

BATTLE_FEATURES = [
    "Gap", "GapClosingRate", "TyreLifeDelta",
    "TrailingCompound", "LeadingCompound",
    "SpeedST_Delta", "CircuitOvertakePrior",
]
BATTLE_CATEGORICAL = ["TrailingCompound", "LeadingCompound"]


def predict_overtake_probability(gap, gap_closing_rate, tyre_life_delta, trailing_compound, leading_compound, speedst_delta, circuit_overtake_prior):
    row = pd.DataFrame([{
        "Gap": gap,
        "GapClosingRate": gap_closing_rate,
        "TyreLifeDelta": tyre_life_delta,
        "TrailingCompound": trailing_compound,
        "LeadingCompound": leading_compound,
        "SpeedST_Delta": speedst_delta,
        "CircuitOvertakePrior": circuit_overtake_prior,
    }])
    for c in BATTLE_CATEGORICAL:
        row[c] = row[c].astype("category")
    return float(battle_model.predict_proba(row[BATTLE_FEATURES])[0, 1])

In [0]:
def predict_battles(year, circuit, lap_number, gap_threshold=3.0):
    race_laps = spark.table("workspace.default.f1_cleaned_lap_dataset").filter(
        (F.col("Year") == year) & (F.col("Circuit") == circuit)
    )

    closing_window = Window.partitionBy("Driver").orderBy("LapNumber").rowsBetween(-2, 0)

    pos_df = race_laps.select(
        "LapNumber", "Driver", "Position", "GapToAhead", "TyreLife", "Compound", "SpeedST"
    ).dropDuplicates(["Driver", "LapNumber"])

    pos_df = (
        pos_df
        .withColumn("_gc_n", F.count("GapToAhead").over(closing_window))
        .withColumn("_gc_sum_x", F.sum("LapNumber").over(closing_window))
        .withColumn("_gc_sum_y", F.sum("GapToAhead").over(closing_window))
        .withColumn("_gc_sum_xy", F.sum(F.col("LapNumber") * F.col("GapToAhead")).over(closing_window))
        .withColumn("_gc_sum_xx", F.sum(F.col("LapNumber") * F.col("LapNumber")).over(closing_window))
    )
    _gc_denom = F.col("_gc_n") * F.col("_gc_sum_xx") - F.col("_gc_sum_x") * F.col("_gc_sum_x")
    pos_df = pos_df.withColumn(
        "gap_closing_rate",
        F.when(
            (F.col("_gc_n") >= 2) & (_gc_denom != 0),
            (F.col("_gc_n") * F.col("_gc_sum_xy") - F.col("_gc_sum_x") * F.col("_gc_sum_y")) / _gc_denom
        ).otherwise(F.lit(None).cast("double"))
    ).drop("_gc_n", "_gc_sum_x", "_gc_sum_y", "_gc_sum_xy", "_gc_sum_xx")

    lap_state = pos_df.filter(F.col("LapNumber") == lap_number)
    trailing = lap_state.alias("t")
    leading = lap_state.alias("l")

    pairs = (
        trailing.join(leading, F.col("t.Position") == F.col("l.Position") + 1)
        .filter((F.col("t.GapToAhead") > 0) & (F.col("t.GapToAhead") <= gap_threshold))
        .select(
            F.col("t.Driver").alias("TrailingDriver"),
            F.col("l.Driver").alias("LeadingDriver"),
            F.col("t.GapToAhead").alias("Gap"),
            F.col("t.gap_closing_rate").alias("GapClosingRate"),
            (F.col("t.TyreLife") - F.col("l.TyreLife")).alias("TyreLifeDelta"),
            F.col("t.Compound").alias("TrailingCompound"),
            F.col("l.Compound").alias("LeadingCompound"),
            (F.col("t.SpeedST") - F.col("l.SpeedST")).alias("SpeedST_Delta"),
        )
        .toPandas()
    )

    prior_row = circuit_overtake_prior.filter(F.col("Circuit") == circuit).first()
    circuit_prior = float(prior_row["CircuitOvertakePrior"]) if prior_row and prior_row["CircuitOvertakePrior"] is not None else 0.15

    battles = []
    for _, row in pairs.iterrows():
        prob = predict_overtake_probability(
            gap=row["Gap"],
            gap_closing_rate=row["GapClosingRate"],
            tyre_life_delta=row["TyreLifeDelta"],
            trailing_compound=row["TrailingCompound"],
            leading_compound=row["LeadingCompound"],
            speedst_delta=row["SpeedST_Delta"],
            circuit_overtake_prior=circuit_prior,
        )
        battles.append({
            "trailing_driver": row["TrailingDriver"],
            "leading_driver": row["LeadingDriver"],
            "gap": float(row["Gap"]),
            "overtake_probability": prob,
        })

    return sorted(battles, key=lambda b: -b["overtake_probability"])

### Post-race replay (Plan E)

Runs `live_model` over every lap of an already-finished race and returns each driver's predicted podium probability trajectory — the cheapest pressure test for whether the lap-by-lap model story is compelling, no live infra required.

In [0]:
FEATURES_LIVE = [
    "LapNumber", "RacePhase", "Position", "GapToAhead", "DeltaToLeader",
    "Compound", "TyreLife", "Stint", "TrackStatus", "tyre_degradation_rate",
    "AirTemp_delta", "TrackTemp_delta", "Humidity_delta", "WindSpeed_delta",
]
CATEGORICAL_LIVE = ["Compound", "RacePhase", "TrackStatus"]


def replay_race(year, circuit):
    race_df = (
        df.filter((F.col("Year") == year) & (F.col("Circuit") == circuit))
        .select(FEATURES_LIVE + ["Driver", "FinalPosition"])
        .toPandas()
    )
    for c in CATEGORICAL_LIVE:
        race_df[c] = race_df[c].astype("category")

    rows = []
    for lap_number, lap_group in race_df.groupby("LapNumber"):
        scores = live_model.predict(lap_group[FEATURES_LIVE])
        exp_scores = np.exp(scores - scores.max())
        probs = exp_scores / exp_scores.sum()
        for driver, final_pos, prob in zip(lap_group["Driver"], lap_group["FinalPosition"], probs):
            rows.append({
                "LapNumber": lap_number,
                "Driver": driver,
                "FinalPosition": final_pos,
                "Probability": prob,
            })

    return pd.DataFrame(rows)